Menghubungkan MongoDB dengan jupyter notebook

In [1]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


menghapus isi database

In [17]:
koleksi_produk.delete_many({})

DeleteResult({'n': 8, 'ok': 1.0}, acknowledged=True)

Create

In [12]:
# hanya menyisipkan 1 data
koleksi_produk.insert_one({
    "nama_produk" : "Daytona Anima",
    "kategori" : "custom performance",
    "harga" : 17000000,
    "stok" : 4,
    "tags" : ["custom engine","160cc"]
})

# menyisipkan lebih banyak data
koleksi_produk.insert_many([
    {"nama_produk" : "Uma Racing ECU" , "kategori": "custom performance" , "harga" : 5000000 , "stok" : 11 , "tags" : ["ECU", "ECU custom"]},
    {"nama_produk" : "KTC Absorber Adjustable" , "kategori" : "custom part" , "harga" : 2000000 , "stok" : 20 , "tags" : ["Shock", "KTC"]},
    {"nama_produk" : "Brembo GP4" , "kategori" : "custom part" , "harga" : 30000000 , "stok" : 3 , "tags" : ["Brembo", "Break"]},
    {"nama_produk" : "Accossato ESC022" , "kategori" : "custom part" , "harga" : 1453324 , "stok" : 2 , "tags" : ["HandBreak", "Accossato"]}
])

print("jumlah data", koleksi_produk.count_documents({}))

jumlah data 5


Read

In [8]:
satu_produk = koleksi_produk.find_one({"kategori": "custom part"})
print("Satu produk kategori custom part:", satu_produk)

print()

# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor, di-looping atau di-list())
print("Seluruh produk kategori custom part:")
for p in koleksi_produk.find({"kategori": "custom part"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

# Filter dengan operator perbandingan
print("Produk dengan harga > 5000000:")
for p in koleksi_produk.find({"harga": {"$gt": 5000000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

Satu produk kategori custom part: {'_id': ObjectId('6ae8b0f9491e1ff571b117df'), 'nama_produk': 'KTC Absorber Adjustable', 'kategori': 'custom part', 'harga': 2000000, 'stok': 20, 'tags': ['Shock', 'KTC']}

Seluruh produk kategori custom part:
  - KTC Absorber Adjustable (Rp2,000,000)
  - Brembo GP4 (Rp30,000,000)
  - Accossato ESC022 (Rp1,453,324)

Produk dengan harga > 5000000:
  - Brembo GP4 (Rp30,000,000)


Update

In [18]:
# update_one() — memperbarui SATU dokumen pertama yang cocok
koleksi_produk.update_one(
    {"nama_produk": "Daytona Anima"},
    {"$set": {"harga": 21000000}}   # menaikkan harga
)

# update_many() — memperbarui SEMUA dokumen yang cocok
koleksi_produk.update_many(
    {"kategori": "custom performance"},
    {"$inc": {"stok": -1}}   # $inc mengubah nilai relatif (increment/decrement)
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

Setelah update:
  - Daytona Anima: harga=Rp21,000,000, stok=0
  - Uma Racing ECU: harga=Rp5,000,000, stok=7
  - Brembo GP4: harga=Rp30,000,000, stok=3
  - Accossato ESC022: harga=Rp1,453,324, stok=2


Delete

In [19]:
# delete_one() — menghapus SATU dokumen pertama yang cocok
koleksi_produk.delete_one({"nama_produk": "KTC Absorber Adjustable"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

Jumlah dokumen tersisa: 4
Daftar produk tersisa: ['Daytona Anima', 'Uma Racing ECU', 'Brembo GP4', 'Accossato ESC022']


Integrasi Data MongoDB ke spark dataframe

In [3]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Tipe data khusus MongoDB (_id, dan kolom berisi list seperti "tags")
# perlu disesuaikan dulu agar Spark dapat membacanya dengan benar
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

26/11/03 10:14:30 WARN Utils: Your hostname, localhost resolves to a loopback address: 127.0.0.1; using 172.18.77.151 instead (on interface wlo1)
26/11/03 10:14:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/03 10:14:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- _id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)



+--------------------+----------------+------------------+--------+----+--------------------+
|                 _id|     nama_produk|          kategori|   harga|stok|                tags|
+--------------------+----------------+------------------+--------+----+--------------------+
|6ae8b34c491e1ff57...|   Daytona Anima|custom performance|21000000|   0|custom engine, 160cc|
|6ae8b34c491e1ff57...|  Uma Racing ECU|custom performance| 5000000|   7|     ECU, ECU custom|
|6ae8b34c491e1ff57...|      Brembo GP4|       custom part|30000000|   3|       Brembo, Break|
|6ae8b34c491e1ff57...|Accossato ESC022|       custom part| 1453324|   2|HandBreak, Accossato|
+--------------------+----------------+------------------+--------+----+--------------------+



In [4]:
from pyspark.sql.functions import avg, count

# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_produk").alias("jumlah_produk")
).show()

spark.stop()
print("SparkSession ditutup.")

+------------------+---------------+-------------+
|          kategori|rata_rata_harga|jumlah_produk|
+------------------+---------------+-------------+
|custom performance|          1.3E7|            2|
|       custom part|    1.5726662E7|            2|
+------------------+---------------+-------------+

SparkSession ditutup.


**Soal 1.** Sisipkan (`insert_many`) lima dokumen baru ke collection `produk` dengan kategori bebas (boleh kategori baru selain Makanan/Minuman, misalnya "Snack").

In [22]:
collections_produk = db["produk"]

collections_produk.insert_many ([
    {"nama_produk" : "Laksa" , "kategori" : "makanan" , "harga" : 20000 , "stok" : 9 , "tags" : ["kuah" , "makanan berat" , "pedas"]},
    {"nama_produk" : "croisant" , "kategori" : "snack" , "harga" : 10000 , "stok" : 30 , "tags" : ["snack" , "roti" , "france"]},
    {"nama_produk" : "ramen" , "kategori" : "makanan" , "harga" : 25000 , "stok" : 15 , "tags" : ["kuah" , "makanan berat" , "mie"]},
    {"nama_produk" : "matcha latte" , "kategori" : "minuman" , "harga" : 10000 , "stok" : 18 , "tags" : ["jepang" , "hijau"]}
])

InsertManyResult([ObjectId('6ae9c32b6686991009fc60ed'), ObjectId('6ae9c32b6686991009fc60ee'), ObjectId('6ae9c32b6686991009fc60ef'), ObjectId('6ae9c32b6686991009fc60f0')], acknowledged=True)

**Soal 2.** Gunakan `find()` dengan operator `$lt` (kurang dari) untuk menampilkan seluruh produk dengan `harga` di bawah 20000.

In [27]:
for f in collections_produk.find({"harga" : {"$lt" : 20000}}):
    print(f'nama barang : {f["nama_produk"]:<16} kategori : {f["kategori"]:<16} harga : {f["harga"]}')

nama barang : croisant         kategori : snack            harga : 10000
nama barang : matcha latte     kategori : minuman          harga : 10000


**Soal 3.** Gunakan `update_many()` untuk menambahkan field baru bernama `diskon` bernilai `0` ke **seluruh** dokumen pada collection `produk` (petunjuk: gunakan operator `$set`).


In [36]:
collections_produk.update_many(
    {},
    {"$set" : {"diskon" : 0}}
)

UpdateResult({'n': 4, 'nModified': 0, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

====================================================================================================================
Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — 
karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya 
hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun 
dan menganalisis katalog ini.


**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

In [22]:
db = client["kopi_nusantara"]

menu = db["produk"]

menu.insert_many ([
    {"nama_menu" : "americano" , "harga" : 10000 , "kategori" : "kopi" , "tingkat_kepahitan" : 3},
    {"nama_menu" : "cappucino" , "harga" : 15000 , "kategori" : "kopi" , "tingkat_kepahitan" : 1},
    {"nama_menu" : "italiano" , "harga" : 10000 , "kategori" : "kopi" , "tingkat_kepahitan" : 5},
    {"nama_menu" : "laksa" , "harga" : 20000 , "kategori" : "makanan" , "tingkat_pedas" : 3},
    {"nama_menu" : "rendang" , "harga" : 50000 , "kategori" : "makanan" , "tingkat_pedas" : 2},
    {"nama_menu" : "shawarma" , "harga" : 18000 , "kategori" : "makanan" , "tingkat_pedas" : 1}
])

InsertManyResult([ObjectId('6aeb2605bd66182706cb3728'), ObjectId('6aeb2605bd66182706cb3729'), ObjectId('6aeb2605bd66182706cb372a'), ObjectId('6aeb2605bd66182706cb372b'), ObjectId('6aeb2605bd66182706cb372c'), ObjectId('6aeb2605bd66182706cb372d')], acknowledged=True)

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

In [23]:
for find in menu.find({"kategori" : "kopi"} and {"tingkat_kepahitan" : {"$gt" : 3}}):
    print("kategori kopi dengan tingkat kepahitan diatas 3 :", find["nama_menu"])

print()

for find2 in menu.find({"harga" : {"$gt" : 30000}} and {"harga" : {"$lt" : 15000}}):
    print("menu diantara harga 15000 sampai 30000 :", find2["nama_menu"])

print()

for find3 in menu.find({"tingkat_pedas" : None}):
    print("Menu yang tidak memiliki tingkat pedas :", find3["nama_menu"])

kategori kopi dengan tingkat kepahitan diatas 3 : italiano

menu diantara harga 15000 sampai 30000 : americano
menu diantara harga 15000 sampai 30000 : italiano

Menu yang tidak memiliki tingkat pedas : americano
Menu yang tidak memiliki tingkat pedas : cappucino
Menu yang tidak memiliki tingkat pedas : italiano


**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

In [24]:
menu.update_many(
    {},
    {"$set" : {"status" : "tersedia"}}
)

menu.update_one(
    {"nama_menu" : "laksa"},
    {"$set" : {"status" : "habis"}}
)
        
menu.update_one(
    {"nama_menu" : "rendang"},
    {"$set" : {"status" : "habis"}}
)

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.


In [29]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.functions import sum as spark_sum, count, avg
import numpy as np

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

c = menu.find({})

doc_mongo = list(c)

panda = pd.DataFrame(doc_mongo)

if '_id' in panda.columns:
    panda['_id'] = panda['_id'].astype(str)

panda['tingkat_kepahitan'] = panda['tingkat_kepahitan'].replace({np.nan: None})
panda['tingkat_pedas'] = panda['tingkat_pedas'].replace({np.nan: None})

spk = spark.createDataFrame(panda)
spk.printSchema()

rata_rata = spk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
)
rata_rata.show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- status: string (nullable = true)
 |-- tingkat_pedas: double (nullable = true)

+--------+------------------+
|kategori|   rata_rata_harga|
+--------+------------------+
|    kopi|11666.666666666666|
| makanan|29333.333333333332|
+--------+------------------+



In [30]:
spk.show()

+--------------------+---------+-----+--------+-----------------+--------+-------------+
|                 _id|nama_menu|harga|kategori|tingkat_kepahitan|  status|tingkat_pedas|
+--------------------+---------+-----+--------+-----------------+--------+-------------+
|6aeb2605bd6618270...|americano|10000|    kopi|              3.0|tersedia|         NULL|
|6aeb2605bd6618270...|cappucino|15000|    kopi|              1.0|tersedia|         NULL|
|6aeb2605bd6618270...| italiano|10000|    kopi|              5.0|tersedia|         NULL|
|6aeb2605bd6618270...|    laksa|20000| makanan|             NULL|   habis|          3.0|
|6aeb2605bd6618270...|  rendang|50000| makanan|             NULL|   habis|          2.0|
|6aeb2605bd6618270...| shawarma|18000| makanan|             NULL|tersedia|          1.0|
+--------------------+---------+-----+--------+-----------------+--------+-------------+



E. Kesimpulan (bobot 20%)

Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

pada awalnya, tidak ada tantangan konversi data dari mongodb ke Spark dataframe secara kode. Namun setelah mencari tau, nilai NaN atau Not a Number akan menghasilkan masalah misalnya pada skenario ketika ingin melakukan operasi matematika atau agregasi misalnya seperti mencari rata-rata atau mencari total, menggunakan Spark SQL pada kolom yang mengandung NaN akan menghasilkan perhitungan yang berantakan atau mengembalikan NaN lagi. Oleh karena itu, data bernilai NaN atau Not as Number dapat diubah nilainya dengan 2 cara yaitu dengan mengubahnya menjadi 0 atau mengubahnya menjadi null. Untuk mengubahnya, pada saat dokumen berada pada format panda, data yang bernilai NaN dapat diubah nilainya sesuai dengan preferensi masing masing atau sesuai ketentuan kode yang berlaku